In [0]:
%python
dbutils.fs.ls("/Volumes/pjmi/inbound/landingzone")

In [0]:
%python
from pyspark.sql import functions as F

df_raw = spark.read.format("text").load("/Volumes/pjmi/inbound/landingzone")

df_withmetada=(df_raw.withColumn("fileName",F.col("_metadata.file_name"))
               .withColumn("FilePath",F.col("_metadata.file_path")))

df_final=(df_withmetada
          .groupBy("filePath")
          .agg(F.concat_ws("\n",F.collect_list("value")).alias("JobText"))
          .withColumn("IngestionTimestamp",F.current_timestamp())
          .withColumn("JobId",F.monotonically_increasing_id())
          .withColumn("JobHash",F.sha2(F.col("JobText"),512))
          .withColumn("JobLength",F.length(F.col("JobText"))))



df_final.write.mode("overwrite").option("mergeSchema","true").saveAsTable("pjmi.bronze.raw_jobs")
       

    







In [0]:
 select * from pjmi.bronze.raw_jobs